In [79]:
import practicalSPARQL
import pandas as pd
import ast
import matplotlib.pyplot as plt
import numpy as np
import bqplot as bq
import geopandas as gpd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.patches as mpatches
import networkx as nx

In [80]:
df = pd.read_csv('../DATA/02_image_clusters/full_image_data_feb_25.csv')
books = pd.read_csv('../DATA/01_corpus_metadata/full_book_data_feb_25.csv')
ucks = pd.read_excel('../DATA/03_content_keywords/all_elements_all_ck_ucks.xlsx')

In [81]:
# Adding printer and publisher data
df['printer'] = df['book'].map(books.set_index('book')['printers'])
df['publisher'] = df['book'].map(books.set_index('book')['publishers'])

# For each image cluster in df, add its uck value from 'ucks' df
df = df.merge(
    ucks[['label', 'uck']],
    how='left',
    left_on='cluster_name',
    right_on='label'
).drop(columns=['label', 'label_x', 'label_y'], errors='ignore')


In [82]:
# Define the target cks values
target_cks = [
   'CK_Structure of the Sublunar World',
   'CK_Qualities of the Aristotelian Elements',
    'CK_Symbols of the Elements',
    'CK_Relation Between the Spheres of Water and Earth'
]

In [83]:
#filtering the df to get the images of the target cks with all their data + printing counts
#(this is why only filter is not enough: we need all the cks that are in other rows with double images values)

# Step 1: Filter the DataFrame for rows where 'cks' is in the target_cks list
filtered_df_target_cks = df[df['cks'].isin(target_cks)]

# Step 2: Get the unique 'images' values associated with the target cks
images_with_target_cks = filtered_df_target_cks['images'].unique()

# Step 3: Filter the original DataFrame to include all rows that have these images
filtered_df = df[df['images'].isin(images_with_target_cks)]

# filtered_df

In [ ]:
#plot number of books with images with relevant cks tags

# Get unique books per year_interval
unique_books_count = books.groupby('year_interval', observed=False)['book'].nunique()

# Get overall number of books including images of all the keywords
overall_keyword_books = df[df['cks'].isin(target_cks)].groupby('year_interval', observed=False)['book'].nunique()

# Get overall number of books with 'Cartography' uck value
cartography_books = df[df['uck'] == 'Cartography'].groupby('year_interval', observed=False)['book'].nunique()

# Define the master timeline from book data
intervals = unique_books_count.index.astype(str)

# Dictionary: {keyword: {interval: unique_book_count}}
book_counts = {
    keyword: (
        df[df['cks'] == keyword]
        .groupby('year_interval', observed=False)['book']
        .nunique()
        .to_dict()
    )
    for keyword in target_cks
}

# Create a DataFrame for the results
book_counts_df = pd.DataFrame(book_counts).fillna(0)

# Add overall number of books with keywords to DataFrame
book_counts_df['Overall Books with Keywords'] = [overall_keyword_books.get(interval, 0) for interval in unique_books_count.index]

# Add cartography books to DataFrame
book_counts_df['Cartography Books'] = [cartography_books.get(interval, 0) for interval in unique_books_count.index]

# Add the total number of books to the DataFrame
book_counts_df['Books'] = unique_books_count.values

# Print the results as a table (zero values will be included)
print("Number of Unique Books per Content Group (CKs) over Time:")
print(book_counts_df)

# Start plotting
plt.figure(figsize=(10, 6))

# Plot books (always included)
plt.plot(intervals, unique_books_count.values, label='Books', marker='o', color='grey', linestyle='--')

# Plot Overall Books with Keywords
overall_keyword_values = [overall_keyword_books.get(interval, 0) for interval in unique_books_count.index]
plt.plot(intervals, overall_keyword_values, label='Overall Books with Keywords', marker='D', color='black', linewidth=2)

# Plot Cartography books
cartography_values = [cartography_books.get(interval, 0) for interval in unique_books_count.index]
plt.plot(intervals, cartography_values, label='Cartography', marker='s', color='red', linestyle='-.', linewidth=2)

# Plot each keyword (number of unique books per target_cks value)
for keyword, data in book_counts.items():
    keyword_values = [data.get(interval, 0) for interval in unique_books_count.index]

    # Exclude zero values from plotting
    non_zero_values = [val for val in keyword_values if val > 0]
    non_zero_intervals = [intervals[i] for i, val in enumerate(keyword_values) if val > 0]

    if non_zero_values:  # Plot only if there are non-zero values
        plt.plot(non_zero_intervals, non_zero_values, marker='o', label=keyword)

# Final touches
plt.title('Number of Unique Books per Content Group (CKs) over Time')
plt.xlabel('Year Interval')
plt.ylabel('Number of Unique Books')
plt.xticks(rotation=45)
plt.legend(title='Keyword', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

# Save plot if needed
# plt.savefig('/Users/nogashlomi/projects/nog_thesis/figures/plots/source_analysis/1.3_books_per_ck_over_time.png', dpi=300)

# Show the plot
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# === PARAMETERS ===

# Define the CK values you want to follow
target_cks = [
    'CK_Terraqueous Globe',
    'CK_Ecumene Conception',
    'CK_Populated Earth',
    'CK_T-O Maps'
]

# === PREPARE DATA ===

# Count total unique books per interval (from 'books' DataFrame)
unique_books_count = books.groupby('year_interval', observed=False)['book'].nunique()

# Create a consistent list of intervals for alignment
intervals = unique_books_count.index.astype(str)

# Count unique books with each CK value per interval (from 'visual_df' DataFrame)
# We filter on df instead of visual_df since the other cells use df
book_counts = {
    keyword: (
        df[df['cks'] == keyword]
        .groupby('year_interval', observed=False)['book']
        .nunique()
        .reindex(unique_books_count.index, fill_value=0)
    )
    for keyword in target_cks
}

# === ADD OVERALL BOOKS WITH ANY OF THE TARGET KEYWORDS ===
overall_keyword_books = (
    df[df['cks'].isin(target_cks)]
    .groupby('year_interval', observed=False)['book']
    .nunique()
    .reindex(unique_books_count.index, fill_value=0)
)

# === PLOTTING ===

plt.figure(figsize=(10, 6))

# Plot total books
plt.plot(intervals, unique_books_count.values, label='All Books', marker='o', color='black', linestyle='--')

# Plot Overall Books with any of the Target Keywords
plt.plot(intervals, overall_keyword_books.values, label='Overall Target Keywords', marker='D', color='black', linewidth=2)

# Plot each CK line (only where value > 0)
for keyword, series in book_counts.items():
    non_zero_intervals = [interval for interval, val in zip(intervals, series.values) if val > 0]
    non_zero_values = [val for val in series.values if val > 0]
    if non_zero_values:
        plt.plot(non_zero_intervals, non_zero_values, marker='o', label=keyword)

# === FORMATTING ===

plt.title('Books with Selected Representations of the Sublunar World')
plt.xlabel('Year Interval')
plt.ylabel('Number of Unique Books')
plt.xticks(rotation=45)
plt.legend(title='CK Tag', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

# Save plot if needed
# plt.savefig('/Users/nogashlomi/projects/nog_thesis/figures/plots/source_analysis/1.3_rep_earth.png', dpi=300)

plt.show()